## Automobile Loan prediction project

In [2]:
#เริ่มจาก import Library ที่ต้องใช้ใน project

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


In [3]:
# โหลดข้อมูลที่ต้องใช้

folder_path = r"C:\Users\Karinthip\Desktop\Automobile Loan Default Dataset"

train_path = folder_path + r"\Train_Dataset.csv"
test_path = folder_path + r"\Test_Dataset.csv"
data_dict_path = folder_path + r"\Data_Dictionary.csv"
sample_submission_path = folder_path + r"\Sample_Submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
data_dict = pd.read_csv(data_dict_path)
sample_submission = pd.read_csv(sample_submission_path)

print(train.shape)
print(test.shape)
print(data_dict.shape)
print(sample_submission.shape)

C:\Users\Karinthip\AppData\Local\Temp\ipykernel_15688\1500321932.py:10: DtypeWarning: Columns (1,7,8,16,17,18,19,20,35) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(train_path)


(121856, 40)
(80900, 39)
(40, 2)
(80900, 2)


C:\Users\Karinthip\AppData\Local\Temp\ipykernel_15688\1500321932.py:11: DtypeWarning: Columns (7,8,16,17,18,19,20,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(test_path)


In [4]:
#เช็คสัดส่วนลูกค้าที่จ่ายหนี้ตรงเวลา(0) กับลูกค้าเบี้ยวหนี้ (1)
print(train["Default"].value_counts(normalize=True))

Default
0    0.919208
1    0.080792
Name: proportion, dtype: float64


In [5]:
#เช็คข้อมูล Missing
print(train.info())

missing = train.isnull().mean().sort_values(ascending=False)
print(missing.head(20))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121856 entries, 0 to 121855
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   ID                          121856 non-null  int64  
 1   Client_Income               118249 non-null  object 
 2   Car_Owned                   118275 non-null  float64
 3   Bike_Owned                  118232 non-null  float64
 4   Active_Loan                 118221 non-null  float64
 5   House_Own                   118195 non-null  float64
 6   Child_Count                 118218 non-null  float64
 7   Credit_Amount               118224 non-null  object 
 8   Loan_Annuity                117044 non-null  object 
 9   Accompany_Client            120110 non-null  object 
 10  Client_Income_Type          118155 non-null  object 
 11  Client_Education            118211 non-null  object 
 12  Client_Marital_Status       118383 non-null  object 
 13  Client_Gender 

In [6]:
# รายชื่อคอลัมน์ที่ต้องเป็นตัวเลขแต่ตอนนี้เป็น object
cols_to_fix = [
    "Client_Income", "Credit_Amount", "Loan_Annuity", 
    "Age_Days", "Employed_Days", "Registration_Days", "ID_Days"
]

for col in cols_to_fix:
    # errors='coerce' จะเปลี่ยนค่าที่แปลงไม่ได้ (เช่น ข้อความแปลกๆ) ให้เป็น NaN
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col] = pd.to_numeric(test[col], errors='coerce')

# ตรวจสอบอีกครั้งว่า Dtype เปลี่ยนเป็น float64 หรือยัง
# print(train[cols_to_fix].dtypes)


In [7]:
def feature_engineering(df):
    df = df.copy()

    # Convert age/employment from days to years
    df["Age_Years"] = df["Age_Days"] / 365
    df["Employment_Years"] = df["Employed_Days"] / 365
    df["Registration_Years"] = df["Registration_Days"] / 365
    df["ID_Years"] = df["ID_Days"] / 365

    # Affordability / DSR features
    df["Loan_to_Income"] = df["Credit_Amount"] / (df["Client_Income"] + 1)
    df["Annuity_to_Income"] = df["Loan_Annuity"] / (df["Client_Income"] + 1)
    df["Credit_to_Annuity"] = df["Credit_Amount"] / (df["Loan_Annuity"] + 1)

    # Family burden
    df["Children_to_Family"] = df["Child_Count"] / (df["Client_Family_Members"] + 1)

    # Missing flags
    important_cols = [
        "Score_Source_1",
        "Score_Source_2",
        "Score_Source_3",
        "Credit_Bureau",
        "Own_House_Age",
        "Social_Circle_Default",
        "Client_Occupation"
    ]

    for col in important_cols:
        if col in df.columns:
            df[col + "_Missing"] = df[col].isnull().astype(int)

    return df


train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121856 entries, 0 to 121855
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   ID                          121856 non-null  int64  
 1   Client_Income               118234 non-null  float64
 2   Car_Owned                   118275 non-null  float64
 3   Bike_Owned                  118232 non-null  float64
 4   Active_Loan                 118221 non-null  float64
 5   House_Own                   118195 non-null  float64
 6   Child_Count                 118218 non-null  float64
 7   Credit_Amount               118219 non-null  float64
 8   Loan_Annuity                117030 non-null  float64
 9   Accompany_Client            120110 non-null  object 
 10  Client_Income_Type          118155 non-null  object 
 11  Client_Education            118211 non-null  object 
 12  Client_Marital_Status       118383 non-null  object 
 13  Client_Gender 

In [9]:
target = "Default"
id_col = "ID"

# Re-create X and X_test_final after feature engineering
X = train_fe.drop(columns=[target, id_col])
y = train_fe[target]

X_test_final = test_fe.drop(columns=[id_col])
test_ids = test_fe[id_col]

# Identify feature types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Convert categorical columns to string for OneHotEncoder
for col in categorical_features:
    X[col] = X[col].astype(str)
    X_test_final[col] = X_test_final[col].astype(str)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Numeric features: 33
Categorical features: 13


In [10]:
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [12]:
logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logit_model.fit(X_train, y_train)

logit_pred_proba = logit_model.predict_proba(X_valid)[:, 1]
logit_auc = roc_auc_score(y_valid, logit_pred_proba)

print("Logistic Regression AUC:", logit_auc)


Logistic Regression AUC: 0.7143357174353622


In [13]:
try:
    from xgboost import XGBClassifier
    xgboost_available = True
except ImportError:
    from sklearn.ensemble import RandomForestClassifier
    xgboost_available = False


In [14]:
if xgboost_available:
    negative_count = (y_train == 0).sum()
    positive_count = (y_train == 1).sum()
    scale_pos_weight = negative_count / positive_count

    xgb_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        ))
    ])

    xgb_model.fit(X_train, y_train)

    xgb_pred_proba = xgb_model.predict_proba(X_valid)[:, 1]
    xgb_auc = roc_auc_score(y_valid, xgb_pred_proba)

    print("XGBoost AUC:", xgb_auc)

else:
    print("XGBoost not installed. Using RandomForest instead.")

    xgb_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    xgb_model.fit(X_train, y_train)

    xgb_pred_proba = xgb_model.predict_proba(X_valid)[:, 1]
    xgb_auc = roc_auc_score(y_valid, xgb_pred_proba)

    print("Random Forest AUC:", xgb_auc)

XGBoost AUC: 0.7280066400814644


In [15]:
precision, recall, thresholds = precision_recall_curve(y_valid, xgb_pred_proba)

threshold_df = pd.DataFrame({
    "threshold": np.append(thresholds, 1),
    "precision": precision,
    "recall": recall
})

print(threshold_df.head())

# ปรับ threshold ตามความ Aggressive / Conservative ของบริษัท
selected_threshold = 0.35

y_valid_pred = (xgb_pred_proba >= selected_threshold).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_valid, y_valid_pred))

print("Classification Report:")
print(classification_report(y_valid, y_valid_pred))


   threshold  precision  recall
0   0.027561   0.080789     1.0
1   0.030228   0.080793     1.0
2   0.034795   0.080796     1.0
3   0.035914   0.080799     1.0
4   0.040079   0.080803     1.0
Confusion Matrix:
[[ 8755 13648]
 [  232  1737]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.39      0.56     22403
           1       0.11      0.88      0.20      1969

    accuracy                           0.43     24372
   macro avg       0.54      0.64      0.38     24372
weighted avg       0.90      0.43      0.53     24372



In [16]:
final_model = xgb_model
final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Client_Income', 'Car_Owned',
                                                   'Bike_Owned', 'Active_Loan',
                                                   'House_Own', 'Child_Count',
                                                   'Credit_Amount',
                                                   'Loan_Annuity', 'Age_Days',
                                                   'Employed_Days',
                                                   'Registration_Days',
                                                   'ID_Days', 'Own_House_Age',
                                                   'M...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.03,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=4, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=500, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [17]:
# แทนที่ค่า # กับ ? ด้วย NaN ในชุด Test

import numpy as np
X_test_final = X_test_final.replace('#', np.nan)
X_test_final = X_test_final.replace('?', np.nan)

#ปรับตัวเลขให้เป็น float อีกรอบ
for col in numeric_features:
    X_test_final[col] = pd.to_numeric(X_test_final[col], errors='coerce')



In [18]:
test_pred_proba = final_model.predict_proba(X_test_final)[:, 1]

result_prediction = pd.DataFrame({
    "ID": test_ids,
    "Default": test_pred_proba
})

result_prediction.to_csv("result_prediction.csv", index=False)

print(result_prediction.head())
print("result_prediction.csv created successfully")

         ID   Default
0  12202227  0.196165
1  12279381  0.761205
2  12222714  0.270348
3  12265215  0.663331
4  12203970  0.402913
result_prediction.csv created successfully
